In [12]:
from pathlib import Path

from lassi.profiler import Timer, MultiProfiler, GPUProfiler, CPUProfiler
from lassi.source_file import SourceFile
from lassi.executer import FunctionalValidator
from typing import Annotated

from openai import AsyncOpenAI
from groq import Groq

from pydantic_ai import Agent
from pydantic_ai.models.openai import OpenAIChatModel
from pydantic_ai.providers.openai import OpenAIProvider

## Minimal C Example

In [ ]:
# Declare a new file

source_file = SourceFile(
    file_name = Path("test.c")
)

source_file

In [ ]:
source_file.write( # this could be the LLM output
    """#include <stdio.h>

        int main(void) {
            printf("Compiler test successful!");
            return 0;
        }
    """
)

executable = source_file.compile()
success = source_file.execute()
report = source_file.get_last_execution_report()

In [ ]:
report # default report is latency

## More complex CUDA example

In [ ]:
source_file = SourceFile(
    file_name = Path("similarity_cuda_test.cu"),
    folder_path = Path("/home/gbrun/test_cuda_folder/src/") # <-- from another folder
)

In [ ]:
source_file # this time lang is CUDA

In [ ]:
source_file.compile(
    kwds="-O3", # all flags go here
    output_file=Path("test_cuda") # custom name
    ) 

In [ ]:
source_file.execute(
    args="100 100", # need args
    profiler=GPUProfiler() # GPU profiler. There is NVIDIA and AMD
    )

In [ ]:
source_file.get_last_execution_report() # more complex report

## Code Generation

In [5]:
import dotenv
from dataclasses import dataclass
from pydantic import BaseModel, Field
from pydantic_ai import Agent
from pydantic_ai.models.groq import GroqModel

from openai import AsyncOpenAI

In [ ]:
dotenv.load_dotenv()

In [6]:
source_file = SourceFile(
    file_name = Path("llm_generated_code.c"),
)

In [8]:
@dataclass
class CodeGenDependencies:
    language: Annotated[str, Field(description="Language to implement the code in.")]

@dataclass
class CodeGenOutput:
    code: Annotated[str, Field(description="Valid code that follows the requested task. Code only.")]

In [ ]:
MODEL_NAME = "openai/gpt-oss-120b"

### Groq model

In [ ]:
model = GroqModel(MODEL_NAME)

### Locally hosted OpenAI (vLLM)

In [ ]:
# model = OpenAIChatModel(MODEL_NAME, provider=OpenAIProvider(base_url="http://localhost:8000/v1"))

### Declare Agent

In [ ]:
agent = Agent(
    model=model,
    instructions=(
        "You are a helpful coding tool."
    ),
    output_type= CodeGenOutput,
)

### Create and test a new file

In [7]:
deps = CodeGenDependencies(language=source_file.lang.value)

result = await agent.run("Write me a C program that takes from args a number N and prints the first N args of the fibonacci sequence.")

source_file.write(result.output.code)

NameError: name 'CodeGenDependencies' is not defined

In [9]:
print(source_file.read())

#include <stdio.h>
#include <stdlib.h>
#include <errno.h>

int main(int argc, char *argv[])
{
    if (argc != 2) {
        fprintf(stderr, "Usage: %s N\n", argv[0]);
        return EXIT_FAILURE;
    }

    char *endptr;
    errno = 0;
    long n_long = strtol(argv[1], &endptr, 10);
    if (errno != 0 || *endptr != '\0' || n_long < 0) {
        fprintf(stderr, "Invalid non‑negative integer: %s\n", argv[1]);
        return EXIT_FAILURE;
    }
    int N = (int)n_long;

    if (N == 0) {
        /* Nothing to print */
        return EXIT_SUCCESS;
    }

    unsigned long long a = 0, b = 1;
    for (int i = 0; i < N; ++i) {
        if (i == 0) {
            printf("%llu", a);
        } else if (i == 1) {
            printf(" %llu", b);
        } else {
            unsigned long long c = a + b;
            printf(" %llu", c);
            a = b;
            b = c;
        }
    }
    printf("\n");
    return EXIT_SUCCESS;
}



In [10]:
source_file.compile()

unit_test = [
    FunctionalValidator(
        args = "4",
        golden_output="0 1 1 2\n",
        ret_code=0),
    FunctionalValidator(
        args = "1",
        golden_output="0\n",
        ret_code=0),
    FunctionalValidator(
        args = "5",
        golden_output="0 1 1 2 3\n",
        ret_code=0),
]

output = source_file.execute(
    validator = unit_test,
)

source_file.get_execution_history()

Compiling /home/gbrun/LASSI-TOOLS/llm_generated_code.c using gcc...
Compiling with command: gcc /home/gbrun/LASSI-TOOLS/llm_generated_code.c -o /home/gbrun/LASSI-TOOLS/llm_generated_code
Running with command: /home/gbrun/LASSI-TOOLS/llm_generated_code 4
Validating output
Running with command: /home/gbrun/LASSI-TOOLS/llm_generated_code 1
Validating output
Running with command: /home/gbrun/LASSI-TOOLS/llm_generated_code 5
Validating output
Passed: 3 / 3


[TimerReport(latency=0.0013724029995501041),
 TimerReport(latency=0.0016458830796182156),
 TimerReport(latency=0.0012260419316589832)]

In [11]:
source_file.compile()

output = source_file.execute(
    args = "10000000",
    profiler=CPUProfiler()
)

source_file.get_execution_history()

Compiling /home/gbrun/LASSI-TOOLS/llm_generated_code.c using gcc...
Compiling with command: gcc /home/gbrun/LASSI-TOOLS/llm_generated_code.c -o /home/gbrun/LASSI-TOOLS/llm_generated_code
Running with command: /home/gbrun/LASSI-TOOLS/llm_generated_code 10000000


[DeviceReport(average_power=25.298, power_peak=25.308, energy=44.129553149700165, memory_usage_avg=0.0, memory_usage_peak=0.0, utilization_avg=0.0, utilization_peak=0.0)]

In [ ]:
cpuprof = CPUProfiler()
cpuprof.start()

In [ ]:
cpuprof.stop()

In [ ]:
list(cpuprof._probe.powers)

In [ ]:
output

In [ ]:
source_file